In [29]:
from pathlib import Path
import json
import shutil
import numpy as np

ROOT = Path("..").resolve()

BASE_CASE = "bird_212_symmetrical_2_1"
BASE_JSONL = ROOT / "data" / "cases" / "sweep" / BASE_CASE / "generated" / f"{BASE_CASE}.jsonl"

CASE_SUFFIXES = list(range(1, 2))   # 11..26 inclusive

MASTER_LIB = ROOT / "data" / "master" / "master_lib.mat"

H, W = 600, 900
MARGIN = 5
SUPERSAMPLE = 4

def ensure_case_jsonl(case_name: str) -> Path:
    out_dir = ROOT / "data" / "cases" / "contrasts" / case_name
    gen_dir = out_dir / "generated"
    gen_dir.mkdir(parents=True, exist_ok=True)

    target = gen_dir / f"{case_name}.jsonl"
    if not target.exists():
        shutil.copy2(BASE_JSONL, target)
    return target

print("Creating cases:")
for s in CASE_SUFFIXES:
    print(" ", f"elephant_7_val_{s}")


Creating cases:
  elephant_7_val_1


In [2]:
import sys
sys.path.append(str(ROOT / "scripts"))

from shape_gen.io_mat import load_generated_case, unit_to_pixel
from shape_gen.library import load_master_records, build_class_index

records = load_master_records(MASTER_LIB)
classes, byClass = build_class_index(records)

print("Loaded records:", len(records))
print("Classes:", len(classes))


Loaded records: 54000
Classes: 54


In [3]:
def load_case_from_jsonl(jsonl_path: Path, default_baseGrid: int = 256):
    row = json.loads(jsonl_path.read_text().splitlines()[-1])

    if "silhouette_u" in row:
        sil_u = np.asarray(row["silhouette_u"], float)
        occ_u = np.asarray(row.get("occluder_u", []), float)
        baseGrid = int(row["baseGrid"])
        sil_class = row.get("sil_class", None)

    else:
        sil_u = np.asarray(row["shape_contour_xy"], float)
        occ_u = np.asarray(row.get("occluder_rect_xy", []), float)
        baseGrid = int(row.get("baseGrid", default_baseGrid))
        sil_class = row.get("sil_class", row.get("category", None))

    if occ_u.size == 0:
        occ_u = np.zeros((0,2), float)

    return sil_u, occ_u, baseGrid, sil_class


In [4]:
# %%
# Cell 4. Worker + merge utilities (FIXED save_xy_npz signature)
from pathlib import Path
import re
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
from shape_gen.generate_parallel2 import generate_completions, save_metadata_jsonl
from shape_gen.heatmap.xy_store import save_xy_npz


def _worker_run(args):
    wid = int(args["worker_id"])
    out_dir = Path(args["out_dir"])
    tmp_dir = Path(args["tmp_dir"])
    tmp_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(int(args["seed"]))

    metas, out_files_xy, polygons_xy, fail_counts = generate_completions(
        silhouette=args["silhouette"],
        occluder=args["occluder"],
        start_pt=args["start_pt"],
        end_pt=args["end_pt"],
        minX=int(args["minX"]),
        minY=int(args["minY"]),
        wBB=int(args["wBB"]),
        hBB=int(args["hBB"]),
        out_w=int(args["out_w"]),
        out_h=int(args["out_h"]),
        out_dir=out_dir,
        silhouette_index=1,
        sil_class=args["sil_class"],
        base_grid=int(args["base_grid"]),
        records=args["records"],
        classes=args["classes"],
        byClass=args["byClass"],
        n_images=int(args["n_attempts"]),
        rng=rng,
        start_index=int(args["start_index"]),
        fraction=float(args["fraction"]),
        final_n_samples_mode=args["final_n_samples_mode"],
        supersample=int(args["supersample"]),
        flush_every=int(args["flush_every"]),
        max_attempts_per_image=int(args["max_attempts_per_image"]),
        require_valid=True,
        snap_intersections_to_vertices=False,
        refit_enabled=False,   # diagnostic run
        refit_n_ctrl=int(args["refit_n_ctrl"]),
        refit_subdiv=int(args["refit_subdiv"]),
        refit_jitter_sigma=float(args["refit_jitter_sigma"]),
        refit_max_attempts=int(args["refit_max_attempts"]),
        shrink_gamma=float(args["shrink_gamma"]),
        max_shrink_iters=int(args["max_shrink_iters"]),
        smooth_win=int(args["smooth_win"]),
        try_mirror=True,
        save_invalid=False,
        invalid_subdir="_invalid",
    )

    meta_path = tmp_dir / f"meta_{wid:02d}.jsonl"
    save_metadata_jsonl(metas, meta_path)

    xy_path = tmp_dir / f"xy_{wid:02d}.npz"
    save_xy_npz(
        xy_path,
        out_files=out_files_xy,
        polygons=polygons_xy,
        base_grid=int(args["base_grid"]),
        matlab_1_indexed=True,
    )

    return str(meta_path), str(xy_path), fail_counts



def extract_idx(p: str) -> int:
    m = re.search(r"completion_\d{4}_(\d{5})\.png$", str(p))
    if not m:
        raise ValueError(f"Could not parse completion index from: {p}")
    return int(m.group(1))


In [5]:
import numpy as np


def _point_in_poly(point, poly):
    x, y = float(point[0]), float(point[1])
    inside = False
    n = len(poly)
    j = n - 1
    for i in range(n):
        xi, yi = float(poly[i, 0]), float(poly[i, 1])
        xj, yj = float(poly[j, 0]), float(poly[j, 1])
        intersects = ((yi > y) != (yj > y)) and (
            x < (xj - xi) * (y - yi) / ((yj - yi) + 1e-12) + xi
        )
        if intersects:
            inside = not inside
        j = i
    return inside


def _closed_seg_lengths(poly):
    nxt = np.roll(poly, -1, axis=0)
    return np.linalg.norm(nxt - poly, axis=1)


def _cum_closed_lengths(poly):
    seg = _closed_seg_lengths(poly)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    return seg, cum, float(seg.sum())


def _project_point_to_closed_poly(poly, p):
    a = poly
    b = np.roll(poly, -1, axis=0)
    ab = b - a
    ap = p[None, :] - a
    denom = np.sum(ab * ab, axis=1)
    denom = np.where(denom < 1e-12, 1e-12, denom)
    t = np.sum(ap * ab, axis=1) / denom
    t = np.clip(t, 0.0, 1.0)
    proj = a + t[:, None] * ab
    d2 = np.sum((proj - p[None, :]) ** 2, axis=1)
    idx = int(np.argmin(d2))
    return idx, float(t[idx]), proj[idx]


def _s_of_projection(poly, idx, t):
    seg, cum, total = _cum_closed_lengths(poly)
    return (cum[idx] + t * seg[idx]) % total


def _point_at_s(poly, s):
    seg, cum, total = _cum_closed_lengths(poly)
    s = s % total
    idx = np.searchsorted(cum, s, side="right") - 1
    idx = min(max(idx, 0), len(seg) - 1)
    ds = s - cum[idx]
    t = 0.0 if seg[idx] < 1e-12 else ds / seg[idx]
    a = poly[idx]
    b = poly[(idx + 1) % len(poly)]
    return a + t * (b - a)


def _forward_arc_len(total, s0, s1):
    return (s1 - s0) % total


def choose_gap_pair_and_fraction(silhouette, occluder, pts):
    """
    Robustly choose the pair of intersections whose arc midpoint lies inside the occluder.
    Returns: start_pt, end_pt, hidden_fraction
    """
    if pts.shape[0] < 2:
        raise RuntimeError(f"Need at least 2 intersections, got {pts.shape[0]}")

    seg, cum, total = _cum_closed_lengths(silhouette)

    # Project all intersection points onto contour arc-length coordinate s
    items = []
    for p in pts:
        idx, t, proj = _project_point_to_closed_poly(silhouette.astype(float), p.astype(float))
        s = _s_of_projection(silhouette.astype(float), idx, t)
        items.append((s, proj))

    items.sort(key=lambda x: x[0])

    # If exactly 2, determine which arc is hidden by midpoint test
    if len(items) == 2:
        s0, p0 = items[0]
        s1, p1 = items[1]

        fwd = _forward_arc_len(total, s0, s1)
        bwd = total - fwd

        mid_fwd = _point_at_s(silhouette.astype(float), s0 + 0.5 * fwd)
        hidden_is_fwd = _point_in_poly(mid_fwd, occluder.astype(float))

        hidden_len = fwd if hidden_is_fwd else bwd
        hidden_fraction = float(np.clip(hidden_len / total, 1e-3, 0.95))
        return np.asarray(p0), np.asarray(p1), hidden_fraction

    # If more than 2, inspect adjacent pairs around the contour
    best = None
    m = len(items)

    for i in range(m):
        s0, p0 = items[i]
        s1, p1 = items[(i + 1) % m]
        arc_len = _forward_arc_len(total, s0, s1)
        mid = _point_at_s(silhouette.astype(float), s0 + 0.5 * arc_len)
        is_hidden = _point_in_poly(mid, occluder.astype(float))

        if is_hidden:
            if best is None or arc_len > best[0]:
                best = (arc_len, p0, p1)

    if best is None:
        # fallback, choose the pair with the largest forward arc
        best_arc = -1
        best_pair = None
        for i in range(m):
            s0, p0 = items[i]
            s1, p1 = items[(i + 1) % m]
            arc_len = _forward_arc_len(total, s0, s1)
            if arc_len > best_arc:
                best_arc = arc_len
                best_pair = (p0, p1)
        hidden_len = best_arc
        p0, p1 = best_pair
    else:
        hidden_len, p0, p1 = best

    hidden_fraction = float(np.clip(hidden_len / total, 1e-3, 0.95))
    return np.asarray(p0), np.asarray(p1), hidden_fraction


In [30]:
# %%
from pathlib import Path
import json
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

from shape_gen.geom_bbox import compute_bbox
from shape_gen.render import draw_and_save
from shape_gen.intersections2 import find_intersection_points_multiple

N_WORKERS = 30
TOTAL_ATTEMPTS = 10_000

SAMPLES_MODE = "matlab_100"

REFIT_N_CTRL = 10
REFIT_SUBDIV = 18
REFIT_JITTER = 0.008
REFIT_MAX_ATTEMPTS = 12

FLUSH = 1000



def _point_in_poly(point, poly):
    x, y = float(point[0]), float(point[1])
    inside = False
    n = len(poly)
    j = n - 1
    for i in range(n):
        xi, yi = float(poly[i, 0]), float(poly[i, 1])
        xj, yj = float(poly[j, 0]), float(poly[j, 1])

        hit = ((yi > y) != (yj > y)) and (
            x < (xj - xi) * (y - yi) / ((yj - yi) + 1e-12) + xi
        )
        if hit:
            inside = not inside
        j = i
    return inside


def _closed_seg_lengths(poly):
    nxt = np.roll(poly, -1, axis=0)
    return np.linalg.norm(nxt - poly, axis=1)


def _cum_closed_lengths(poly):
    seg = _closed_seg_lengths(poly)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    total = float(seg.sum())
    return seg, cum, total


def _project_point_to_closed_poly(poly, p):
    a = poly
    b = np.roll(poly, -1, axis=0)
    ab = b - a
    ap = p[None, :] - a

    denom = np.sum(ab * ab, axis=1)
    denom = np.where(denom < 1e-12, 1e-12, denom)

    t = np.sum(ap * ab, axis=1) / denom
    t = np.clip(t, 0.0, 1.0)

    proj = a + t[:, None] * ab
    d2 = np.sum((proj - p[None, :]) ** 2, axis=1)

    idx = int(np.argmin(d2))
    return idx, float(t[idx]), proj[idx]


def _s_of_projection(poly, idx, t):
    seg, cum, total = _cum_closed_lengths(poly)
    return (cum[idx] + t * seg[idx]) % total


def _point_at_s(poly, s):
    seg, cum, total = _cum_closed_lengths(poly)
    s = s % total

    idx = np.searchsorted(cum, s, side="right") - 1
    idx = min(max(idx, 0), len(seg) - 1)

    ds = s - cum[idx]
    t = 0.0 if seg[idx] < 1e-12 else ds / seg[idx]

    a = poly[idx]
    b = poly[(idx + 1) % len(poly)]
    return a + t * (b - a)


def _forward_arc_len(total, s0, s1):
    return (s1 - s0) % total


def choose_gap_pair_and_fraction(silhouette, occluder, pts):
    """
    Robustly pick the pair of intersections that bound the actually occluded arc.
    Returns:
        start_pt, end_pt, hidden_fraction
    """
    if pts.shape[0] < 2:
        raise RuntimeError(f"Need at least 2 intersections, got {pts.shape[0]}")

    silhouette = np.asarray(silhouette, dtype=float)
    occluder = np.asarray(occluder, dtype=float)
    pts = np.asarray(pts, dtype=float)

    _, _, total = _cum_closed_lengths(silhouette)

    items = []
    for p in pts:
        idx, t, proj = _project_point_to_closed_poly(silhouette, p)
        s = _s_of_projection(silhouette, idx, t)
        items.append((s, proj))

    items.sort(key=lambda x: x[0])

    # Common case: exactly two intersections
    if len(items) == 2:
        s0, p0 = items[0]
        s1, p1 = items[1]

        fwd = _forward_arc_len(total, s0, s1)
        bwd = total - fwd

        mid_fwd = _point_at_s(silhouette, s0 + 0.5 * fwd)
        hidden_is_fwd = _point_in_poly(mid_fwd, occluder)

        hidden_len = fwd if hidden_is_fwd else bwd
        hidden_fraction = float(np.clip(hidden_len / max(total, 1e-12), 1e-3, 0.95))

        return np.asarray(p0), np.asarray(p1), hidden_fraction

    # More complex case: >2 intersections
    best = None
    m = len(items)

    for i in range(m):
        s0, p0 = items[i]
        s1, p1 = items[(i + 1) % m]

        arc_len = _forward_arc_len(total, s0, s1)
        mid = _point_at_s(silhouette, s0 + 0.5 * arc_len)
        is_hidden = _point_in_poly(mid, occluder)

        if is_hidden:
            if best is None or arc_len > best[0]:
                best = (arc_len, p0, p1)

    if best is None:
        # fallback: choose the largest arc between adjacent intersections
        best_arc = -1.0
        best_pair = None
        for i in range(m):
            s0, p0 = items[i]
            s1, p1 = items[(i + 1) % m]
            arc_len = _forward_arc_len(total, s0, s1)
            if arc_len > best_arc:
                best_arc = arc_len
                best_pair = (p0, p1)

        hidden_len = best_arc
        p0, p1 = best_pair
    else:
        hidden_len, p0, p1 = best

    hidden_fraction = float(np.clip(hidden_len / max(total, 1e-12), 1e-3, 0.95))
    return np.asarray(p0), np.asarray(p1), hidden_fraction


for suffix in CASE_SUFFIXES:

    CASE = f"bird_212_symmetrical_2_{suffix}"
    print("\n=== CASE:", CASE)

    jsonl_path = ensure_case_jsonl(CASE)

    OUT_DIR = ROOT / "data" / "cases" / "sweep" / CASE
    RAND_DIR = OUT_DIR / "generated" / "completions"
    TMP = OUT_DIR / "generated" / "_tmp"

    RAND_DIR.mkdir(parents=True, exist_ok=True)
    TMP.mkdir(parents=True, exist_ok=True)

    sil_u, occ_u, baseGrid, sil_class = load_case_from_jsonl(jsonl_path)

    silhouette = unit_to_pixel(sil_u, baseGrid)
    occluder = unit_to_pixel(occ_u, baseGrid) if occ_u.size else np.zeros((0, 2), dtype=np.int32)

    polys = [silhouette] + ([occluder] if occluder.size else [])
    minX, minY, wBB, hBB = compute_bbox(
        polys,
        base_grid=baseGrid,
        margin=MARGIN
    )

    # Write GT
    draw_and_save(
        polygons=[silhouette],
        colors=[[0, 0, 0]],
        minX=minX, minY=minY, wBB=wBB, hBB=hBB,
        out_w=W, out_h=H,
        out_file=OUT_DIR / "gt.png",
        supersample=SUPERSAMPLE
    )

    # Write occluded
    if occluder.size:
        draw_and_save(
            polygons=[silhouette, occluder],
            colors=[[0, 0, 0], [131, 131, 131]],
            minX=minX, minY=minY, wBB=wBB, hBB=hBB,
            out_w=W, out_h=H,
            out_file=OUT_DIR / "occluded.png",
            supersample=SUPERSAMPLE
        )
    else:
        print("Occluder empty. Skipping occluded render.")

    if not occluder.size:
        raise RuntimeError(f"{CASE}: occluder is empty. Cannot generate completions without intersections.")

    pts = find_intersection_points_multiple(silhouette, occluder, eps_merge=1e-3)
    if pts.shape[0] < 2:
        raise RuntimeError(f"{CASE}: <2 intersection points found. Got {pts.shape[0]}")

    start_pt, end_pt, case_fraction = choose_gap_pair_and_fraction(
        silhouette.astype(float),
        occluder.astype(float),
        pts.astype(float),
    )

    # Adaptive parameters based on actual hidden contour fraction
    sil_bb = silhouette.max(axis=0) - silhouette.min(axis=0)
    sil_diag = float(np.linalg.norm(sil_bb))
    gap_px = float(np.linalg.norm(end_pt - start_pt))
    gap_rel = gap_px / max(sil_diag, 1e-6)

    CASE_FRACTION = float(np.clip(case_fraction, 0.03, 0.80))

    CASE_MAX_ATTEMPTS = int(np.clip(
        250 + 2500 * abs(CASE_FRACTION - 0.22),
        250,
        4000
    ))

    CASE_SHRINK_GAMMA = float(np.clip(
        0.78 + 0.20 * CASE_FRACTION,
        0.78,
        0.96
    ))

    CASE_MAX_SHRINK_ITERS = int(np.clip(
        60 + 180 * CASE_FRACTION,
        60,
        220
    ))

    CASE_SMOOTH_WIN = int(np.clip(
        2 * round(3 + 12 * CASE_FRACTION) + 1,
        5,
        21
    ))

    print(
        f"{CASE}: hidden_fraction={CASE_FRACTION:.4f}, "
        f"gap_px={gap_px:.2f}, gap_rel={gap_rel:.4f}, "
        f"max_attempts={CASE_MAX_ATTEMPTS}, "
        f"shrink_gamma={CASE_SHRINK_GAMMA:.3f}, "
        f"max_shrink_iters={CASE_MAX_SHRINK_ITERS}, "
        f"smooth_win={CASE_SMOOTH_WIN}"
    )

    CASE_SEED = int(suffix)

    # Distribute attempts with remainder
    base_attempts = TOTAL_ATTEMPTS // N_WORKERS
    remainder = TOTAL_ATTEMPTS % N_WORKERS

    jobs = []
    next_start_index = 1  # matlab_1_indexed=True downstream

    for wid in range(N_WORKERS):

        n_attempts_w = base_attempts + (1 if wid < remainder else 0)
        worker_seed = CASE_SEED * 1_000_000 + wid * 10_000 + 123

        jobs.append(dict(
            worker_id=wid,
            seed=worker_seed,
            tmp_dir=str(TMP),
            out_dir=str(RAND_DIR),

            silhouette=silhouette,
            occluder=occluder,
            start_pt=start_pt,
            end_pt=end_pt,
            minX=minX,
            minY=minY,
            wBB=wBB,
            hBB=hBB,
            out_w=W,
            out_h=H,
            sil_class=sil_class,
            base_grid=baseGrid,

            records=records,
            classes=classes,
            byClass=byClass,

            n_attempts=n_attempts_w,
            start_index=next_start_index,

            fraction=CASE_FRACTION,
            final_n_samples_mode=SAMPLES_MODE,
            supersample=SUPERSAMPLE,
            flush_every=FLUSH,
            max_attempts_per_image=CASE_MAX_ATTEMPTS,

            refit_n_ctrl=REFIT_N_CTRL,
            refit_subdiv=REFIT_SUBDIV,
            refit_jitter_sigma=REFIT_JITTER,
            refit_max_attempts=REFIT_MAX_ATTEMPTS,

            shrink_gamma=CASE_SHRINK_GAMMA,
            max_shrink_iters=CASE_MAX_SHRINK_ITERS,
            smooth_win=CASE_SMOOTH_WIN,
        ))

        next_start_index += n_attempts_w

    assert next_start_index == TOTAL_ATTEMPTS + 1, (next_start_index, TOTAL_ATTEMPTS)

    results = []
    all_fail_counts = []

    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = [ex.submit(_worker_run, j) for j in jobs]
        for fut in as_completed(futures):
            try:
                results.append(fut.result(timeout=20))
            except Exception as e:
                print("Worker failed:", repr(e))
                raise


    from collections import Counter

    fail_counter = Counter()
    for _, _, fc in results:
        fail_counter.update(fc)

    print("AGGREGATED FAIL COUNTS:")
    for k, v in sorted(fail_counter.items()):
        print(f"  {k}: {v}")

    # Merge meta
    all_meta = []
    for m, _, _ in results:
        all_meta += Path(m).read_text(encoding="utf-8").splitlines()

    meta_rows = [json.loads(l) for l in all_meta if l.strip()]
    meta_rows.sort(key=lambda r: int(r.get("completion_index", 0)))

    meta_out = OUT_DIR / "generated" / "shapes_meta.jsonl"
    meta_out.parent.mkdir(exist_ok=True, parents=True)

    with meta_out.open("w", encoding="utf-8") as f:
        for r in meta_rows:
            f.write(json.dumps(r) + "\n")

    # Merge xy
    pairs = []
    for _, xy, _ in results:
        npz = np.load(xy, allow_pickle=True)
        for fpath, poly in zip(npz["out_files"], npz["polygons"]):
            pairs.append((extract_idx(fpath), str(fpath), poly))

    pairs.sort(key=lambda t: t[0])

    out_files = [p[1] for p in pairs]
    polys_xy = [p[2] for p in pairs]

    xy_out = OUT_DIR / "generated" / "shapes_xy.npz"
    save_xy_npz(
        xy_out,
        out_files=out_files,
        polygons=polys_xy,
        base_grid=int(baseGrid),
        matlab_1_indexed=True,
    )

    print("Saved:", CASE, " valid:", len(out_files))



=== CASE: bird_212_symmetrical_2_1
bird_212_symmetrical_2_1: hidden_fraction=0.0535, gap_px=20.55, gap_rel=0.0791, max_attempts=666, shrink_gamma=0.791, max_shrink_iters=69, smooth_win=9
[generate] completion_index=1 / 334
[generate] completion_index=6 / 334
[generate] completion_index=11 / 334
[generate] completion_index=16 / 334
[generate] completion_index=335 / 668
[generate] completion_index=21 / 334
[generate] completion_index=340 / 668
[generate] completion_index=26 / 334
[generate] completion_index=345 / 668
[generate] completion_index=31 / 334
[generate] completion_index=350 / 668
[generate] completion_index=36 / 334
[generate] completion_index=355 / 668
[generate] completion_index=669 / 1002
[generate] completion_index=41 / 334
[generate] completion_index=360 / 668
[generate] completion_index=674 / 1002
[generate] completion_index=46 / 334
[generate] completion_index=365 / 668
[generate] completion_index=51 / 334
[generate] completion_index=679 / 1002
[generate] completion_in